In [3]:
# %cd ..
import rectools
import pandas as pd
import more_itertools
import hydra
import torch
import torch.nn as nn
from rectools.model_selection.time_split import TimeRangeSplitter
from rectools.metrics.classification import Precision, Recall
from rectools.metrics.ranking import MAP, NDCG
from rectools.metrics.diversity import IntraListDiversity
from rectools.metrics.novelty import MeanInvUserFreq
from rectools.metrics.serendipity import Serendipity
from rectools.models import (
    PopularInCategoryModel,
    PopularModel,
    RandomModel,
    ImplicitItemKNNWrapperModel,
    ImplicitALSWrapperModel,
)
from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender
# from implicit.cpu.als import AlternatingLeastSquares
from rectools.dataset.dataset import Dataset
from rectools.metrics import calc_metrics
import numpy as np
from rectools import Columns
from rectools.model_selection import cross_validate
from implicit.gpu.als import AlternatingLeastSquares
import implicit

In [4]:
RANDOM_STATE = 42
TOP_N = 10
NUM_THREADS = 20

metrics = {
    "Prec@10": Precision(k=10),
    "Recall@10": Recall(k=10),
    "MAP@10": MAP(k=10),
    "NDCG@10": NDCG(k=10),}

In [5]:
user_df = pd.read_csv("data/users_processed.csv")
items_df = pd.read_csv("data/items_processed.csv")
interactions_df = pd.read_csv("data/interactions_processed.csv",  parse_dates=['last_watch_dt'])
user_df

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,M,True
1,962099,age_18_24,income_20_40,M,False
2,1047345,age_45_54,income_40_60,F,False
3,721985,age_45_54,income_20_40,F,False
4,704055,age_35_44,income_60_90,F,False
...,...,...,...,...,...
840192,339025,age_65_inf,income_0_20,F,False
840193,983617,age_18_24,income_20_40,F,True
840194,251008,age_unknown,income_unknown,sex_unknown,False
840195,590706,age_unknown,income_unknown,F,False


In [6]:
user_features = ["age", "income", "sex", "kids_flg"]
item_features = ["countries", "for_kids", "age_rating", "studios", "release_year_cat"]

In [7]:
def flatten_df(df, features, id_name):
    feature_frames = []
    for feature in features:
        frame = df.reindex(columns=[id_name, feature])
        frame.columns = ["id", "value"]
        frame["feature"] = feature
        feature_frames.append(frame)
    df_flatten = pd.concat(feature_frames)
    return df_flatten


user_df_flatten = flatten_df(user_df, user_features, "user_id")
user_df_flatten

,id,value,feature
0,973171,age_25_34,age
1,962099,age_18_24,age
2,1047345,age_45_54,age
3,721985,age_45_54,age
4,704055,age_35_44,age
...,...,...,...
840192,339025,False,kids_flg
840193,983617,True,kids_flg
840194,251008,False,kids_flg
840195,590706,False,kids_flg


In [8]:
interactions = interactions_df.rename(
    columns={"watched_pct": "weight", "last_watch_dt": "datetime"}
)
interactions

,user_id,item_id,datetime,total_dur,weight
0,176549,9506,2021-05-11,4250,72
1,699317,1659,2021-05-29,8317,100
2,656683,7107,2021-05-09,10,0
3,864613,7638,2021-07-05,14483,100
4,964868,9506,2021-04-30,6725,100
...,...,...,...,...,...
5476246,648596,12225,2021-08-13,76,0
5476247,546862,9673,2021-04-13,2308,49
5476248,697262,15297,2021-08-20,18307,63
5476249,384202,16197,2021-04-19,6203,100


In [9]:
item_df_flatten = flatten_df(items_df, item_features, "item_id")
item_df_flatten

,id,value,feature
0,10711,испания,countries
1,2508,сша,countries
2,10716,канада,countries
3,7868,великобритания,countries
4,16268,ссср,countries
...,...,...,...
15958,6443,2010-2020,release_year_cat
15959,2367,2020_inf,release_year_cat
15960,10632,2010-2020,release_year_cat
15961,4538,2010-2020,release_year_cat


In [10]:
ds = Dataset.construct(
    interactions_df=interactions,
    user_features_df=user_df_flatten,
    cat_user_features=user_features,
    item_features_df=item_df_flatten,
    cat_item_features=item_features,
)
ds

Dataset(user_id_map=IdMap(external_ids=array([176549, 699317, 656683, ..., 365945, 983617, 166555], dtype=int64)), item_id_map=IdMap(external_ids=array([ 9506,  1659,  7107, ...,  7536, 12939,  1830], dtype=int64)), interactions=Interactions(df=         user_id  item_id  weight   datetime
0              0        0    72.0 2021-05-11
1              1        1   100.0 2021-05-29
2              2        2     0.0 2021-05-09
3              3        3   100.0 2021-07-05
4              4        0   100.0 2021-04-30
...          ...      ...     ...        ...
5476246   962177      208     0.0 2021-08-13
5476247   224686     2690    49.0 2021-04-13
5476248   962178       21    63.0 2021-08-20
5476249     7934     1725   100.0 2021-04-19
5476250   631989      157    45.0 2021-08-15

[5476251 rows x 4 columns]), user_features=SparseFeatures(values=<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 3360788 stored elements and shape (1058088, 19)>, names=(('age', 'age_25_34'), ('age', 

In [11]:
splitter = TimeRangeSplitter(
    test_size="7D",
    n_splits=3,
    filter_cold_users=False,
    filter_cold_items=False,
    filter_already_seen=False,
)

In [ ]:
def fit_cv_model(dataset, model, splitter, top_n, metrics):
    metrics_df = pd.DataFrame()
    for fold, (train_ids, test_ids, _) in enumerate(
        splitter.split(dataset.interactions)
    ):
        train_ds = dataset.filter_interactions(train_ids)
        interactions_df_test = dataset.interactions.df.loc[test_ids]
        interactions_df_train = dataset.interactions.df.loc[train_ids]
        interactions_df_test[Columns.User] = dataset.user_id_map.convert_to_external(interactions_df_test[Columns.User])
        interactions_df_test[Columns.Item] = dataset.item_id_map.convert_to_external(interactions_df_test[Columns.Item])
        interactions_df_train[Columns.User] = dataset.user_id_map.convert_to_external(interactions_df_train[Columns.User])

        hot_test_users = np.intersect1d(interactions_df_test[Columns.User].unique(), interactions_df_train[Columns.User].unique())
        model.fit(train_ds)
        recs = model.recommend(
            users = hot_test_users,
            dataset = train_ds,
            k = top_n,
            filter_viewed = True,
            on_unsupported_targets="warn",
        )
        metr = calc_metrics(
                    metrics=metrics,
                    reco=recs,
                    interactions=interactions_df_test.loc[interactions_df_test["user_id"].isin(hot_test_users)],
                )
        cur_metrics = pd.DataFrame(
            [
                metr
            ]
        )
        cur_metrics["fold"] = fold
        metrics_df = pd.concat([metrics_df, cur_metrics])
        print(f"eval fold: {fold}")
        print(metrics_df)
    return metrics_df, model, recs

In [ ]:
model = ImplicitALSWrapperModel(
    AlternatingLeastSquares(
        factors=256,  # latent embeddings size
        regularization=0.5,
        iterations=10,
        alpha=10,  # confidence multiplier for non-zero entries in interactions
        random_state=RANDOM_STATE,
        num_threads=NUM_THREADS,
    ),
    fit_features_together=False,  # way to fit paired features
)
metrics_df, model, recomencations = fit_cv_model(
    ds, model, splitter, TOP_N, metrics=metrics
)

c:\Users\user\AppData\Local\pypoetry\Cache\virtualenvs\ods-recsys-competition-kzVvw0Ap-py3.10\lib\site-packages\rectools\dataset\features.py:424: UserWarning: Converting sparse features to dense array may cause MemoryError
  warnings.warn("Converting sparse features to dense array may cause MemoryError")
100%|██████████| 1/1 [00:56<00:00, 56.46s/it]


{'Prec@10': 0.024943661971830983, 'Recall@10': 0.12378253832090415, 'NDCG@10': 0.02787533508409996, 'MAP@10': 0.04778959370354875}
eval fold: 0
    Prec@10  Recall@10   NDCG@10   MAP@10  fold
0  0.024944   0.123783  0.027875  0.04779     0
